# RAG - Ingestion
This code reads the data file about_me.md from the assets folder and then breaks it into chunks.
1. Reads the data present in the document and load it into memory
2. Chunks the data
3. Embed the chunks and persist it into a vector data store using Chroma

## Read the directory and load the documents

In [2]:
from pathlib import Path
from langchain_core.documents import Document

def load_documents(directory):
    documents = []

    for file_path in Path(directory).rglob("*"):
        if file_path.suffix.lower() not in [".md"]:
            continue

        if not file_path.is_file():
            continue

        content = file_path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": str(file_path),
                    "filename": file_path.name
                }
            )
        )

    return documents


docs = load_documents("../assets/about_me")

print(f"Loaded {len(docs)} documents")

Loaded 1 documents


## Chunk the loaded data from documents

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks")

Created 18 chunks


## Data Embedding and Storing in Chroma

Pull the nomic-embed-text model for ollama using the below command

ollama pull nomic-embed-text

Verify the model using the ollama list command

ollama list
NAME                       ID              SIZE      MODIFIED
nomic-embed-text:latest    0a109f422b47    274 MB    4 minutes ago
qwen3:8b                   500a1f067a9f    5.2 GB    4 weeks ago


The code creates an embedding object reference to the ollama -> nomic-embed-text

Then it take the data chunks and uses the embedding object create dabove to embedd the data chunks and store it in the chroma DB and persist in the location

In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

db_name = "../assets/about_me/vector_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

The below code gives a metadata of how many vectors are created with how many dimensions

In [7]:
collection = vectorstore._collection
count = collection.count()
print(collection.name)

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

langchain
There are 54 vectors with 768 dimensions in the vector store
